Compare three type of GEMM multiplications:

[x x x x x] * [w w w w w w]  
[x x x x 0 0 0 x x x x] * [w w w w 0 0 0 w w w w] and  
[x x x 0 0 0] * [w w w 0 0 0]

for three types: bf16, fp16 and fp32.

"different" - num of elements in tensor that not equal.

In [9]:
! python /glazkov-dev/LoRa-Transfer-Pruning/experiments/compare_tp_and_our/test_full_vs_compact_gemm.py \
    --device cuda:0

GPU: NVIDIA A100-PCIE-40GB
Float32 matmul precision: highest

torch.bfloat16 (compact K=3968, full K=4096, internal gap at 1000)
compact vs full with internal zeros
  equal:     True
  different: 0
  max:       0.00000000e+00
  mean:      0.00000000e+00
  rmse:      0.00000000e+00
compact vs full with trailing zeros
  equal:     True
  different: 0
  max:       0.00000000e+00
  mean:      0.00000000e+00
  rmse:      0.00000000e+00

torch.float16 (compact K=3968, full K=4096, internal gap at 1000)
compact vs full with internal zeros
  equal:     True
  different: 0
  max:       0.00000000e+00
  mean:      0.00000000e+00
  rmse:      0.00000000e+00
compact vs full with trailing zeros
  equal:     True
  different: 0
  max:       0.00000000e+00
  mean:      0.00000000e+00
  rmse:      0.00000000e+00

torch.float32 (compact K=3968, full K=4096, internal gap at 1000)
compact vs full with internal zeros
  equal:     False
  different: 60154
  max:       1.22070312e-04
  mean:      1.79389262

In [10]:
! python /glazkov-dev/LoRa-Transfer-Pruning/experiments/compare_tp_and_our/test_full_vs_compact_gemm.py \
    --device cuda:0 \
    --batch-size 1

GPU: NVIDIA A100-PCIE-40GB
Float32 matmul precision: highest

torch.bfloat16 (compact K=3968, full K=4096, internal gap at 1000)
compact vs full with internal zeros
  equal:     True
  different: 0
  max:       0.00000000e+00
  mean:      0.00000000e+00
  rmse:      0.00000000e+00
compact vs full with trailing zeros
  equal:     True
  different: 0
  max:       0.00000000e+00
  mean:      0.00000000e+00
  rmse:      0.00000000e+00

torch.float16 (compact K=3968, full K=4096, internal gap at 1000)
compact vs full with internal zeros
  equal:     True
  different: 0
  max:       0.00000000e+00
  mean:      0.00000000e+00
  rmse:      0.00000000e+00
compact vs full with trailing zeros
  equal:     True
  different: 0
  max:       0.00000000e+00
  mean:      0.00000000e+00
  rmse:      0.00000000e+00

torch.float32 (compact K=3968, full K=4096, internal gap at 1000)
compact vs full with internal zeros
  equal:     False
  different: 3369
  max:       4.57763672e-05
  mean:      8.02452269e

In [11]:
! python /glazkov-dev/LoRa-Transfer-Pruning/experiments/compare_tp_and_our/test_full_vs_compact_gemm.py \
    --device cuda:0     --batch-size 1 --full-k=4096 --compact-k=3857

GPU: NVIDIA A100-PCIE-40GB
Float32 matmul precision: highest

torch.bfloat16 (compact K=3857, full K=4096, internal gap at 1000)
compact vs full with internal zeros
  equal:     False
  different: 9
  max:       5.00000000e-01
  mean:      1.74522400e-04
  rmse:      7.99560547e-03
compact vs full with trailing zeros
  equal:     False
  different: 8
  max:       5.00000000e-01
  mean:      1.74522400e-04
  rmse:      7.99560547e-03

torch.float16 (compact K=3857, full K=4096, internal gap at 1000)
compact vs full with internal zeros
  equal:     False
  different: 52
  max:       1.25000000e-01
  mean:      1.72615051e-04
  rmse:      3.11660767e-03
compact vs full with trailing zeros
  equal:     False
  different: 52
  max:       1.25000000e-01
  mean:      1.56164169e-04
  rmse:      2.95066833e-03

torch.float32 (compact K=3857, full K=4096, internal gap at 1000)
compact vs full with internal zeros
  equal:     False
  different: 3451
  max:       6.10351562e-05
  mean:      8.157

Conclusion: sometimes bf16 and fp16 show no difference for small enough gap_size (num of zeroes), but fp32 shows diff.  
For higher gap size we see difference in all types (fp32, fp16, bf16).

Also diff can be caused by different batch_size.

Internal zeroes more often leads to DIFF than trailing zeroes.

FP32 can show diff, where bf16 and fp16 can be "equal".

Result depends on shape and distribution of zeroes, and concrete tensor splitting for GEMM.